## Initialization

### Retrieve Parameters

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
    
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        # if aws_secret_key is not None:
        #     print("AWS secret key retrieved successfully.")
        # else:
        #     print("AWS secret key is None. Please check Databricks secrets configuration.")

        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration

        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    print("mode is shared")
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

### Input Parameters

**report_args:** it is a json which contains all the appropriate parameters to run the report

`
{   
    "portfolio_name": ""MONZA PORTFOLIO - RETAIL",
    "output_file_prefix": "Monza_Retail_Daily_Extraction_3_",
    "report_output": "/Volumes/qa-cdh-delivery/internalreports/transactions-exports/DataExtraction3/MonzaRetail/"
}
`

**run_date:** is the date value to run the report. The run_date is empty when it is called by the orchestrator and that means "today"

In [0]:
from typing import Any
import shutil
import os

from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional

import delta  # type: ignore
from pyspark.sql import SparkSession

from io import BytesIO
import pandas as pd
import json

from datetime import datetime
from pytz import timezone

In [0]:
athens_tz = timezone('Europe/Athens')

try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("'", '"')
except Exception as e:
    raise ValueError("Missing required input: report_args")

try:    
    report_name = dbutils.widgets.get("report_name")
except Exception as e:
    raise ValueError("Missing required input: report_name")
print(f"Report {report_name} is started")

try:
    run_date = dbutils.widgets.get("run_date")
    if run_date is None or run_date == "": raise ValueError("ERROR: run_date has no value. Remove the run_date completely if you want to run the job using the current date or add a value in the following format 'YYYY-MM-DD'")
    run_date_str = run_date.replace('-', '')
    run_date = datetime.strptime(run_date, '%Y-%m-%d').date()
except Exception as e:
    run_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    run_date_str = datetime.now(athens_tz).strftime('%Y%m%d')

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

run_date = run_date.strftime('%Y-%m-%d')

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "portfolio_name" not in json_data: raise KeyError("'portfolio_name' key is missing in the report_args JSON data.")
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

portfolioName = "".join(json_data["portfolio_name"])
output_file_prefix = "".join(json_data["output_file_prefix"])
report_output = "".join(json_data["report_output"])

print(f"Report {report_name} is running with the following arguments: report_args = {report_args}, report_name = {report_name}, portfolio_name = {portfolioName}, output_file_prefix = {output_file_prefix}, run_date = {run_date}")

print(f"now is: {now_datetime_str}")

In [0]:
RefMonth = spark.sql(f"SELECT CAST(DATEADD(DAY, 1 - DAY('{run_date}'), CAST('{run_date}' AS DATE)) AS DATE)").collect()[0][0]

#Overwrite temporarily the values for testing purposes - DEBUG - To be deleted
output_file_prefix = "TransactionsExport_3_"
#report_output = "/Volumes/qa-cdh-delivery/internalreports/transactions-exports"

## Report Generation 

### Application fields

In [0]:
applications = spark.sql(f"""
                                select 
                                    PortfolioName, 
                                    BusinessDate as Date,
                                    SubmittedDate, 
                                    CDI, 
                                    ApplicationStatus,
                                    CancelledDate, 
                                    FlowSegment, 
                                    CreatedBy, 
                                    UserCreatedGroup, 
                                    ApplicationKey,
                                    BusinessDate, 
                                    SourceSystem
                                from `{env}-cdh`.primary.application
                                where   BusinessDate = {run_date!r}
                                    and PortfolioName ilike '%{portfolioName}%'
                                    /*(   PortfolioName ILIKE '%MONZA PORTFOLIO - RETAIL%'
                                        or  PortfolioName ILIKE '%DOMUS PORTFOLIO - RETAIL%'
                                        or  PortfolioName ILIKE '%RHODIUM PORTFOLIO - RETAIL%')*/
                            """)

applications.createOrReplaceTempView("applications")
                             

## Report export

In [0]:
final_report_df = applications.toPandas()

In [0]:
# Define the temporary file and output path
temp_file = f"/tmp/{output_file_prefix}{now_datetime_str}.xlsx"

output_buffer = BytesIO()
with pd.ExcelWriter(output_buffer) as writer:
    final_report_df.to_excel(writer, sheet_name=f"{RefMonth.month}.{RefMonth.year}", index=False)

with open(temp_file, "wb") as f:
    f.write(output_buffer.getvalue())

output_path = f"{report_output}{run_date_str}/{output_file_prefix}{run_date}.xlsx"

# Ensure the directory exists
dbutils.fs.mkdirs(f"{report_output}{run_date_str}")

# Delete the output path if it exists
if os.path.exists(output_path):
    os.remove(output_path)

# Copy the temporary file to the final output path
shutil.copy(temp_file, output_path)

print(f"Output file is written successfully: {output_path}")

## Debug

In [0]:
%sql
--select distinct PortfolioName, SourceSystem from applications%sql
--select * from `p-cdh`.primary.application where applicationkey in ('129868', '126313') and BusinessDate = '2025-03-31'

In [0]:
%sql
--select ApplicationKey, CDI, count(*)
--from applications
--where ApplicationKey is not null 
--group by ApplicationKey, CDI
--order by count(*) desc
--limit 5